In [1]:
from query import load_sqgdp
import pandas as pd

In [2]:
df = load_sqgdp(2,'CA')

In [3]:
df.head()

,GeoFIPS,GeoName,Region,TableName,LineCode,IndustryClassification,Description,Unit,value,period
0,06000,California,8.0,SQGDP2,1,...,All industry total,Millions of current dollars,1657672.4,2005Q1
1,06000,California,8.0,SQGDP2,1,...,All industry total,Millions of current dollars,1682374.6,2005Q2
2,06000,California,8.0,SQGDP2,1,...,All industry total,Millions of current dollars,1716355.1,2005Q3
3,06000,California,8.0,SQGDP2,1,...,All industry total,Millions of current dollars,1739031.4,2005Q4
4,06000,California,8.0,SQGDP2,1,...,All industry total,Millions of current dollars,1788491.4,2006Q1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2295 entries, 0 to 2294
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype        
---  ------                  --------------  -----        
 0   GeoFIPS                 2295 non-null   str          
 1   GeoName                 2295 non-null   str          
 2   Region                  2295 non-null   float64      
 3   TableName               2295 non-null   str          
 4   LineCode                2295 non-null   int64        
 5   IndustryClassification  2295 non-null   str          
 6   Description             2295 non-null   str          
 7   Unit                    2295 non-null   str          
 8   value                   2295 non-null   float64      
 9   period                  2295 non-null   period[Q-DEC]
dtypes: float64(2), int64(1), period[Q-DEC](1), str(6)
memory usage: 358.6 KB


# GeoFIPS

In [5]:
df_geofips = df.copy()

# check whether every GeoFIPS value is made up of digits only (i.e. represents an integer)
is_all_integer = df_geofips['GeoFIPS'].str.fullmatch(r'\d+').all()
print("GeoFIPS is all integer values:", is_all_integer)

GeoFIPS is all integer values: True


# GeoName

In [6]:
df_geoname = df.copy()

print("Unique GeoName values:")
print(df_geoname['GeoName'].unique())

# make sure neither GeoFIPS nor GeoName has null values
print("\nGeoFIPS null count:", df_geoname['GeoFIPS'].isnull().sum())
print("GeoName null count:", df_geoname['GeoName'].isnull().sum())

Unique GeoName values:
<ArrowStringArray>
['California']
Length: 1, dtype: str

GeoFIPS null count: 0
GeoName null count: 0


# Region

In [7]:
df_region = df.copy()

print("Unique Region values:")
print(df_region['Region'].unique())

# make sure Region is not all null
print("\nRegion null count:", df_region['Region'].isnull().sum())
print("Region is all null:", df_region['Region'].isnull().all())

Unique Region values:
[8.]

Region null count: 0
Region is all null: False


# TableName

In [8]:
df_tablename = df.copy()

print("Unique TableName values:")
print(df_tablename['TableName'].unique())

# TableName should be a single, consistent value across all rows
is_single_value = df_tablename['TableName'].nunique() == 1
print("\nTableName is all the same value:", is_single_value)

Unique TableName values:
<ArrowStringArray>
['SQGDP2']
Length: 1, dtype: str

TableName is all the same value: True


# LineCode

The industry/measure code for the row. Cross-reference against the table's `__definition.xml` to get its label. For SQGDP2/8/9/11, common codes include: `1` all industry total, `2` private industries, `83` government and government enterprises, `84` federal civilian, `85` military, `86` state and local, plus one code per NAICS-based industry (agriculture, mining, construction, manufacturing, retail trade, finance and insurance, health care, etc.). SQGDP1 only has 3 summary lines (no industry breakdown). Each state/geo appears as multiple rows in a file — one row per `LineCode`.

In [9]:
df_linecode = df.copy()

print("Unique LineCode values:")
print(sorted(df_linecode['LineCode'].unique()))

Unique LineCode values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(6), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(25), np.int64(34), np.int64(35), np.int64(36), np.int64(45), np.int64(51), np.int64(56), np.int64(60), np.int64(64), np.int64(65), np.int64(69), np.int64(70), np.int64(76), np.int64(79), np.int64(82), np.int64(83), np.int64(84), np.int64(85), np.int64(86)]


# IndustryClassification

The NAICS code(s) for that `LineCode`'s industry (e.g. `11` for agriculture), or `"..."` for summary lines that aren't a single NAICS sector (e.g. all-industry total, private industries, government). Effectively a NAICS-coded companion to `LineCode`/`Description` — use it when you need the standard NAICS classification rather than BEA's internal line numbering.

In [10]:
df_industry = df.copy()

print("Unique IndustryClassification values:")
print(df_industry['IndustryClassification'].unique())

# check whether LineCode and IndustryClassification have null values
print("\nLineCode null count:", df_industry['LineCode'].isnull().sum())
print("IndustryClassification null count:", df_industry['IndustryClassification'].isnull().sum())

Unique IndustryClassification values:
<ArrowStringArray>
[            '...',              '11',              '21',              '22',
              '23',           '31-33',     '321,327-339', '311-316,322-326',
              '42',           '44-45',           '48-49',              '51',
              '52',              '53',              '54',              '55',
              '56',              '61',              '62',              '71',
              '72',              '81',              '92']
Length: 23, dtype: str

LineCode null count: 0
IndustryClassification null count: 0


### Lookup dictionaries: LineCode and IndustryClassification meanings

Built from `SQGDP2__definition.xml` (BEA's official line definitions) plus the actual `LineCode`/`IndustryClassification` pairs observed in the SQGDP2 CSVs.

Note: `LineCode` is BEA's own line numbering — it does **not** match the NAICS code in `IndustryClassification`. For example `LineCode == 51` is *Finance and insurance*, while `IndustryClassification == "51"` (NAICS 51) is *Information*. Don't assume the two are interchangeable.

In [11]:
# LineCode -> human-readable label (from SQGDP2__definition.xml)
LINECODE_LABELS = {
    1: "All industry total",
    2: "Private industries",
    3: "Agriculture, forestry, fishing and hunting",
    6: "Mining, quarrying, and oil and gas extraction",
    10: "Utilities",
    11: "Construction",
    12: "Manufacturing",
    13: "Durable goods manufacturing",
    25: "Nondurable goods manufacturing",
    34: "Wholesale trade",
    35: "Retail trade",
    36: "Transportation and warehousing",
    45: "Information",
    51: "Finance and insurance",
    56: "Real estate and rental and leasing",
    60: "Professional, scientific, and technical services",
    64: "Management of companies and enterprises",
    65: "Administrative and support and waste management and remediation services",
    69: "Educational services",
    70: "Health care and social assistance",
    76: "Arts, entertainment, and recreation",
    79: "Accommodation and food services",
    82: "Other services (except government and government enterprises)",
    83: "Government and government enterprises",
    84: "Federal civilian",
    85: "Military",
    86: "State and local",
    100: "All industry total, overseas activity",
    101: "Government and government enterprises, overseas activity",
}

# IndustryClassification (NAICS code) -> sector label.
# Summary lines (LineCode 1, 2, 83, 84, 85, 86) use "..." instead of a NAICS
# code since they aren't a single sector, so "..." is intentionally excluded here.
INDUSTRY_CLASSIFICATION_LABELS = {
    "11": "Agriculture, forestry, fishing and hunting",
    "21": "Mining, quarrying, and oil and gas extraction",
    "22": "Utilities",
    "23": "Construction",
    "31-33": "Manufacturing",
    "321,327-339": "Durable goods manufacturing",
    "311-316,322-326": "Nondurable goods manufacturing",
    "42": "Wholesale trade",
    "44-45": "Retail trade",
    "48-49": "Transportation and warehousing",
    "51": "Information",
    "52": "Finance and insurance",
    "53": "Real estate and rental and leasing",
    "54": "Professional, scientific, and technical services",
    "55": "Management of companies and enterprises",
    "56": "Administrative and support and waste management and remediation services",
    "61": "Educational services",
    "62": "Health care and social assistance",
    "71": "Arts, entertainment, and recreation",
    "72": "Accommodation and food services",
    "81": "Other services (except government and government enterprises)",
    "92": "Government and government enterprises",
}

# example: map the loaded df's LineCode/IndustryClassification to labels
df_labeled = df.copy()
df_labeled['LineCodeLabel'] = df_labeled['LineCode'].map(LINECODE_LABELS)
df_labeled['IndustryLabel'] = df_labeled['IndustryClassification'].map(INDUSTRY_CLASSIFICATION_LABELS)
df_labeled[['LineCode', 'LineCodeLabel', 'IndustryClassification', 'IndustryLabel']].drop_duplicates()

,LineCode,LineCodeLabel,IndustryClassification,IndustryLabel
0,1,All industry total,...,NaN
85,2,Private industries,...,NaN
170,3,"Agriculture, forestry, fishing and hunting",11,"Agriculture, forestry, fishing and hunting"
255,6,"Mining, quarrying, and oil and gas extraction",21,"Mining, quarrying, and oil and gas extraction"
340,10,Utilities,22,Utilities
425,11,Construction,23,Construction
510,12,Manufacturing,31-33,Manufacturing
595,13,Durable goods manufacturing,"321,327-339",Durable goods manufacturing
680,25,Nondurable goods manufacturing,"311-316,322-326",Nondurable goods manufacturing
765,34,Wholesale trade,42,Wholesale trade


# Description

In [12]:
df_description = df.copy()

# does Description have null values?
print("Description null count:", df_description['Description'].isnull().sum())

Description null count: 0


# Unit

In [13]:
df_unit = df.copy()

print("Unique Unit values:")
print(df_unit['Unit'].unique())

# can Unit be converted to float?
try:
    df_unit['Unit'].astype(float)
    print("\nUnit can be converted to float")
except ValueError as e:
    print("\nUnit cannot be converted to float:", e)

Unique Unit values:
<ArrowStringArray>
['Millions of current dollars']
Length: 1, dtype: str

Unit cannot be converted to float: could not convert string to float: 'Millions of current dollars'


# period

In [14]:
df_period = df.copy()

# range of period
print("period min:", df_period['period'].min())
print("period max:", df_period['period'].max())
print("period null count:", df_period['period'].isnull().sum())

period min: 2005Q1
period max: 2026Q1
period null count: 0


### What is Dtype `period[Q-DEC]`?

`period[Q-DEC]` is pandas' `PeriodDtype` for **quarterly periods where the year ends in December** (i.e. standard calendar quarters: Q1 = Jan-Mar, Q2 = Apr-Jun, Q3 = Jul-Sep, Q4 = Oct-Dec).

- Each value is a `pandas.Period` object (e.g. `2005Q1`), not a `Timestamp` — it represents a fixed span of time (the whole quarter) rather than a single point in time.
- Unlike a plain string like `"2005Q1"`, a `Period` supports date arithmetic (`+ 1` moves to the next quarter), sorting, comparisons, and resampling/grouping by quarter.
- The `-DEC` suffix indicates the fiscal year-end month used to anchor the quarters; `Q-DEC` aligns with the standard calendar year.

# Download

In [18]:
import zipfile
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen

ZIP_URL = 'https://apps.bea.gov/regional/zip/SQGDP.zip'

# for py file
# CURRENT_DIR = Path(__file__).resolve().parent

CURRENT_DIR = Path.cwd()

output_folder = CURRENT_DIR/"SQGDP"

print("downloading folder")

with urlopen(ZIP_URL) as response:
    zip_buffer = BytesIO(response.read())

    with zipfile.ZipFile(zip_buffer) as zip_file:
        for file_info in zip_file.infolist():
            if file_info.filename.endswith(".csv") and not file_info.filename.startswith("__MACOSX"):
                print(f" Extracting: {file_info.filename}")
                zip_file.extract(file_info, path=output_folder)




downloading folder
 Extracting: SQGDP11_SWST_2005_2026.csv
 Extracting: SQGDP11_RKMT_2005_2026.csv
 Extracting: SQGDP11_FWST_2005_2026.csv
 Extracting: SQGDP1__ALL_AREAS_2005_2026.csv
 Extracting: SQGDP2_US_2005_2026.csv
 Extracting: SQGDP2__ALL_AREAS_2005_2026.csv
 Extracting: SQGDP2_AL_2005_2026.csv
 Extracting: SQGDP2_AK_2005_2026.csv
 Extracting: SQGDP2_AZ_2005_2026.csv
 Extracting: SQGDP2_AR_2005_2026.csv
 Extracting: SQGDP2_CA_2005_2026.csv
 Extracting: SQGDP2_CO_2005_2026.csv
 Extracting: SQGDP2_CT_2005_2026.csv
 Extracting: SQGDP2_DE_2005_2026.csv
 Extracting: SQGDP2_DC_2005_2026.csv
 Extracting: SQGDP2_FL_2005_2026.csv
 Extracting: SQGDP2_GA_2005_2026.csv
 Extracting: SQGDP2_HI_2005_2026.csv
 Extracting: SQGDP2_ID_2005_2026.csv
 Extracting: SQGDP2_IL_2005_2026.csv
 Extracting: SQGDP2_IN_2005_2026.csv
 Extracting: SQGDP2_IA_2005_2026.csv
 Extracting: SQGDP2_KS_2005_2026.csv
 Extracting: SQGDP2_KY_2005_2026.csv
 Extracting: SQGDP2_LA_2005_2026.csv
 Extracting: SQGDP2_ME_2005_202

# Transformation

In [29]:
from query import load_sqgdp

df = load_sqgdp('SQGDP2','US')

df.head()



,GeoFIPS,GeoName,Region,TableName,LineCode,IndustryClassification,Description,Unit,value,period
0,00000,United States *,,SQGDP2,1,...,All industry total,Millions of current dollars,12767286.0,2005Q1
1,00000,United States *,,SQGDP2,1,...,All industry total,Millions of current dollars,12922656.0,2005Q2
2,00000,United States *,,SQGDP2,1,...,All industry total,Millions of current dollars,13142642.0,2005Q3
3,00000,United States *,,SQGDP2,1,...,All industry total,Millions of current dollars,13324204.0,2005Q4
4,00000,United States *,,SQGDP2,1,...,All industry total,Millions of current dollars,13599160.0,2006Q1
